In [ ]:
import os
import sys
import pathlib

# set to root directory
if 'changed_root' not in locals():
    os.chdir(os.path.dirname(os.getcwd()))
    changed_root = True
print(os.getcwd())


# Brainwear Pipeline

Replicates the CIFAR-10 pretrain -> AACBR -> contest -> global-optimize pipeline on wearable accelerometer data.

## Preprocessing

In [ ]:
! python -m scripts.preprocess_brainwear --window-epochs 15 --test-frac 0.2

In [ ]:
import json

from IPython.display import Image, display

from deeparguing.output_paths import find_output

with open("data/brainwear/window_meta.json") as f:
    manifest = json.load(f)
display(manifest)

display(Image(filename=find_output("brainwear_window_label_distribution.png")))

## Pretrain LSTM embedding

In [ ]:
! python -m scripts.pretrain_lstm --epochs 20 --hidden-size 64 --embedding-size 32 --window-epochs 15

In [ ]:
from IPython.display import Image, display

from deeparguing.output_paths import find_output

display(Image(filename=find_output("lstm_brainwear_per_class_f1.png")))
print("Checkpoint:", find_output("lstm_brainwear.pt"))

## Deep Arguing Model

In [ ]:
! python src/deeparguing/cli/run.py --config "tuning/brainwear/data_brainwear.yaml" "tuning/brainwear/hyperparameters_brainwear.yaml" "tuning/brainwear/model_brainwear.yaml" --seed "(0, 5)" --log "info" --run_train -lv --run_test --misclassified_log --grae_log

In [ ]:
from IPython.display import Markdown, display

from deeparguing.output_paths import find_output

with open(find_output("summary.md")) as f:
    display(Markdown(f.read()))

## Contest

In [ ]:
! python -m deeparguing.contest.scripts.contest_all

In [ ]:
! python -m deeparguing.contest.scripts.run_global_contest_eval

## Globally Optimize

In [ ]:
! python -m deeparguing.contest.global_optimize

In [ ]:
from IPython.display import Markdown, display

from deeparguing.output_paths import find_output

with open(find_output("global_contest_eval.md")) as f:
    display(Markdown(f.read()))
with open(find_output("global_optimize.md")) as f:
    display(Markdown(f.read()))